# **SpaceX Falcon 9 First Stage Landing Prediction**


## Web Scraping Falcon 9 and Falcon Heavy Launches Records from Wikipedia


In this notebook, I'll be performing web scraping to collect Falcon 9 historical launch records from a Wikipedia page titled `List of Falcon 9 and Falcon Heavy launches`

https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches


## Project Goals
My objectives for this web scraping portion of the project are to:
- Extract a Falcon 9 launch records HTML table from Wikipedia
- Parse the table and convert it into a Pandas data frame for further analysis


In [2]:
import sys

import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd

I'll define some helper functions to process the web scraped HTML table data:


In [3]:
def date_time(table_cells):
    """
    This function returns the data and time from the HTML table cell
    Input: the element of a table data cell extracts extra row
    """
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    """
    This function returns the booster version from the HTML table cell 
    Input: the element of a table data cell extracts extra row
    """
    out=''.join([booster_version for i,booster_version in enumerate( table_cells.strings) if i%2==0][0:-1])
    return out

def landing_status(table_cells):
    """
    This function returns the landing status from the HTML table cell 
    Input: the element of a table data cell extracts extra row
    """
    out=[i for i in table_cells.strings][0]
    return out


def get_mass(table_cells):
    mass=unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass.find("kg")
        new_mass=mass[0:mass.find("kg")+2]
    else:
        new_mass=0
    return new_mass


def extract_column_from_header(row):
    """
    This function returns the landing status from the HTML table cell 
    Input: the element of a table data cell extracts extra row
    """
    if (row.br):
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
        
    colunm_name = ' '.join(row.contents)
    
    # Filter the digit and empty names
    if not(colunm_name.strip().isdigit()):
        colunm_name = colunm_name.strip()
        return colunm_name    


To ensure consistent data for my analysis, I'll be working with a snapshot of the `List of Falcon 9 and Falcon Heavy launches` Wikipedia page updated on `9th June 2021`


In [4]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

Next, I'll request the HTML page from the URL and get a `response` object


## Requesting the Falcon 9 Launch Wiki Page


First, I'll perform an HTTP GET method to request the Falcon 9 Launch HTML page as an HTTP response.


In [5]:
# Use requests.get() method with the provided static_url
response = requests.get(static_url)

Now I'll create a `BeautifulSoup` object from the HTML `response`


In [6]:
# Use BeautifulSoup() to create a BeautifulSoup object from the response text content
soup = BeautifulSoup(response.text)

I'll print the page title to verify if the `BeautifulSoup` object was created properly 


In [7]:
# Use soup.title attribute to verify the page title
soup.title

<title>List of Falcon 9 and Falcon Heavy launches - Wikipedia</title>

## Extracting Column Names from the HTML Table Header


Next, I need to collect all relevant column names from the HTML table header


Let me first find all tables on the wiki page


In [8]:
# Use the find_all function in the BeautifulSoup object to find all tables
html_tables = soup.find_all("table")

The third table is my target table containing the actual launch records.


In [9]:
# Access the third table which contains launch records
first_launch_table = html_tables[2]
#print(first_launch_table)

Now I'll iterate through the `<th>` elements and apply the `extract_column_from_header()` function to extract column names one by one


In [10]:
column_names = []

# Find all th elements and extract column names
for header in first_launch_table.find_all("th"):
    col_name = extract_column_from_header(header)
    if col_name: column_names.append(col_name)

Let's check the extracted column names


In [11]:
print(column_names)

['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']


## Creating a DataFrame by Parsing the Launch HTML Tables


I'll create an empty dictionary with keys from the extracted column names. Later, this dictionary will be converted into a Pandas dataframe


In [12]:
launch_dict = dict.fromkeys(column_names)

# Remove an irrelevant column
del launch_dict['Date and time ( )']

# Initialize the launch_dict with each value as an empty list
launch_dict['Flight No.'] = []
launch_dict['Launch site'] = []
launch_dict['Payload'] = []
launch_dict['Payload mass'] = []
launch_dict['Orbit'] = []
launch_dict['Customer'] = []
launch_dict['Launch outcome'] = []
# Added some new columns
launch_dict['Version Booster']=[]
launch_dict['Booster landing']=[]
launch_dict['Date']=[]
launch_dict['Time']=[]

Now I need to fill the `launch_dict` with launch records extracted from table rows.


HTML tables in Wiki pages often contain unexpected annotations and other types of noise, such as reference links `B0004.1[8]`, missing values `N/A [e]`, and inconsistent formatting. I'll need to handle these carefully during parsing.


In [13]:
extracted_row = 0
# Extract each table 
for table_number, table in enumerate(soup.find_all('table', "wikitable plainrowheaders collapsible")):
    # Get table rows 
    for rows in table.find_all("tr"):
        # Check to see if first table heading is a number corresponding to launch number 
        if rows.th:
            if rows.th.string:
                flight_number = rows.th.string.strip()
                flag = flight_number.isdigit()
        else:
            flag = False
        # Get table elements 
        row = rows.find_all('td')
        # If it is a numbered row, save cells in the dictionary 
        if flag:
            extracted_row += 1
            # Flight Number value
            launch_dict["Flight No."].append(flight_number)
            
            # Extract date and time
            datatimelist = date_time(row[0])
            
            # Date value
            date = datatimelist[0].strip(',')
            launch_dict["Date"].append(date)
            
            # Time value
            time = datatimelist[1]
            launch_dict["Time"].append(time)
              
            # Booster version
            bv = booster_version(row[1])
            if not(bv):
                bv = row[1].a.string
            launch_dict["Version Booster"].append(bv)
            
            # Launch Site
            launch_site = row[2].a.string
            launch_dict["Launch site"].append(launch_site)
            
            # Payload
            payload = row[3].a.string
            launch_dict["Payload"].append(payload)
            
            # Payload Mass
            payload_mass = get_mass(row[4])
            launch_dict["Payload mass"].append(payload_mass)
            
            # Orbit
            orbit = row[5].a.string
            launch_dict["Orbit"].append(orbit)
            
            # Customer
            if row[6].a == None:
                customer = row[6].string
            else:
                customer = row[6].a.string
            launch_dict["Customer"].append(customer)
            
            # Launch outcome
            launch_outcome = list(row[7].strings)[0]
            launch_dict["Launch outcome"].append(launch_outcome)
            
            # Booster landing
            booster_landing = landing_status(row[8])
            launch_dict["Booster landing"].append(booster_landing)

Now that I've filled the `launch_dict` with parsed launch record values, I can create a dataframe from it.


In [14]:
df = pd.DataFrame({key:pd.Series(value) for key, value in launch_dict.items()})

Let's export this data to a CSV file for further analysis in the next notebook:

In [1]:
#df.to_csv('data/spacex_web_scraped.csv', index=False)